# Chapter 9 &mdash; Exponential Blow-Up in NFA-to-RE Conversion

**Concept 3 of the Chapter 9 decomposition:** *Exponential Blow-Up in NFA-to-RE Conversion*

A graph of size $N$ can have $O(2^N)$ paths, and the RE must name every one of them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Exponential-Blow-Up-NFA2RE/Concept-Exponential-Blow-Up-NFA2RE.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The produced regular expression can be **exponentially longer** than the NFA. The
reason is structural: a graph with $N$ states can have $O(2^N)$ distinct paths, and an
RE that denotes the language must, in effect, **name them all**.

Each elimination step multiplies label lengths ($m\times n$ bypasses, each
concatenating three labels), so length grows multiplicatively down the sequence.

Two practical consequences:

* the **order** of deletion matters a great deal in practice &mdash; Jove's
  `choose_state_to_del` picks a state with few edges;
* the RE you get is rarely the shortest one; simplification is a separate problem (and
  finding the shortest RE is hard).

## 2. Definitions

### A family of machines whose RE grows fast

In [ ]:
def diamond(k):
    """k diamonds in series -- each doubles the number of paths"""
    lines = ['NFA']
    for i in range(k):
        a, b = 'S%d' % i, 'S%d' % (i+1)
        src = 'I' if i == 0 else a
        dst = 'F' if i == k-1 else b
        lines.append('%s : 0 -> U%d' % (src, i))
        lines.append('%s : 1 -> V%d' % (src, i))
        lines.append('U%d : 0 -> %s' % (i, dst))
        lines.append('V%d : 1 -> %s' % (i, dst))
    return md2mc('\n'.join(lines))

### Convert and measure

In [ ]:
def re_of(N, dellist=None):
    g = mk_gnfa(N)
    _, _, r = del_gnfa_states(g) if dellist is None else del_gnfa_states(g, DelList=dellist)
    return r

## 3. Tests

Paths double with each diamond; so does the RE length.

In [ ]:
rows = []
for k in range(1, 5):
    N = diamond(k)
    r = re_of(N)
    rows.append((k, len(N["Q"]), 2**k, len(r)))
    print("k=%d : |Q| = %2d, paths = %3d, RE length = %4d" % rows[-1])
assert rows[-1][3] > rows[0][3] * 4, "RE length should grow fast"

Every produced RE is still **correct** &mdash; long, not wrong.

In [ ]:
for k in range(1, 4):
    N = diamond(k)
    r = re_of(N)
    assert iso_dfa(min_dfa(nfa2dfa(N)), min_dfa(nfa2dfa(re2nfa(r))))
    print("k=%d : round-trips to an isomorphic minimal DFA" % k)

**Deletion order matters.** Two orders, two very different lengths.

In [ ]:
N = diamond(3)
inner = sorted(q for q in N["Q"] if q.startswith(('U', 'V')))
outer = sorted(q for q in N["Q"] if not q.startswith(('U', 'V')))
# a DelList must be a permutation of ALL original states
assert sorted(inner + outer) == sorted(N["Q"])
r1 = re_of(N, inner + outer)
r2 = re_of(N, outer + inner)
print("inner-first : RE length %d" % len(r1))
print("outer-first : RE length %d" % len(r2))
assert iso_dfa(min_dfa(nfa2dfa(re2nfa(r1))), min_dfa(nfa2dfa(re2nfa(r2))))
print("same language either way :", True)

The minimal DFA, by contrast, stays small &mdash; the blow-up is in the *notation*.

In [ ]:
for k in range(1, 5):
    N = diamond(k)
    print("k=%d : minimal DFA %2d states, RE %4d characters"
          % (k, len(min_dfa(nfa2dfa(N))["Q"]), len(re_of(N))))

## 4. Exercises


1. Why does a graph with $N$ states have up to $2^N$ paths?
2. Design a deletion order for `diamond(4)` that beats both of the ones above.
3. Is finding the shortest equivalent RE decidable? Is it tractable?

In [ ]:
# Your work for the exercises above.